# Steady-State Density Contour Plot ($U$ vs IHF)

Computes the integrated heat flux (IHF) for Holton-Mass and Emulator data, then produces a side-by-side log-density contour plot in physical units.

Requires long-run data files ($\geq$1M steps):
- HM simulation: `long_run_3mil.npy`
- Emulator output: `pred_3mil.npy`

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300

In [3]:
# Configuration
HM_DATA_PATH = 'long_run_3mil.npy'
EM_DATA_PATH = 'pred_3mil.npy'
SAVE_PATH = os.path.join('..', 'graphs_for_paper', 'steady_state.png')

TIMESTEPS = 1_000_000
LEVEL = 63

## Physical Constants & Helper Functions

Constants from the Holton-Mass model. IHF computation adapted from [Finkel et al. (2021)](https://github.com/justinfocus12/SHORT).

In [4]:
q = {
    'rad': 6370.0e3, 'g': 9.82, 'phi0': np.pi / 3,
    'sx': 2, 'zB_d': 0.0, 'zT_d': 70.0e3, 'H': 7.0e3,
    'Omega': 2 * np.pi / (24 * 3600), 'Nsq_d': 4.0e-4,
    'ideal_gas_constant': 287.0, 'hB_d': 38.5,
    'Nz': 26, 'length': 2.5e5, 'time': 24 * 3600.0,
}

n_pts = q['Nz'] + 1
q['f0_d'] = 2 * q['Omega'] * np.sin(q['phi0'])
q['k_d'] = q['sx'] / (q['rad'] * np.cos(q['phi0']))
q['dz_d'] = (q['zT_d'] - q['zB_d']) / q['Nz']
q['z_d'] = np.linspace(q['zB_d'], q['zT_d'], n_pts)
q['Psi0_d'] = q['g'] * q['hB_d'] / q['f0_d']
q['fn'] = q['f0_d'] ** 2 / q['Nsq_d']
q['k'] = q['k_d'] * q['length']
q['dz'] = q['dz_d'] / q['H']
q['z'] = q['z_d'] / q['H']
q['Psi0'] = q['Psi0_d'] * q['time'] / q['length'] ** 2

In [6]:
def first_derivative(F, lower, upper, dz):
    """Central finite-difference first derivative on interior grid."""
    Nt, n = F.shape
    Fz = np.zeros([Nt, n])
    Fz[:, 1:-1] = (F[:, 2:n] - F[:, 0:n-2]) / (2 * dz)
    Fz[:, 0] = (F[:, 1] - lower) / (2 * dz)
    Fz[:, -1] = (upper - F[:, -2]) / (2 * dz)
    return Fz


def ihf_function(data):
    """Integrated meridional heat flux (IHF) on the HM grid."""
    n = q['Nz'] - 1
    x = data
    Nt = len(x)
    heat_flux = np.ones([Nt, n + 1])
    heat_flux *= q['k']

    Xz = first_derivative(x[:, :n], q['Psi0'], 0, q['dz'])
    Yz = first_derivative(x[:, n:2*n], 0, 0, q['dz'])
    Yz0 = (4 * x[:, 2*n] - x[:, 2*n+1]) / (2 * q['dz'])
    heat_flux[:, 1:] *= x[:, :n] * Yz - x[:, n:2*n] * Xz
    heat_flux[:, 0] *= q['Psi0'] * Yz0
    heat_flux *= np.exp(-q['z'][:-1])

    ihf = np.zeros((Nt, n))
    ihf[:, 0] = 0.5 * (heat_flux[:, 0] + heat_flux[:, 1]) * q['dz']
    for i in range(1, n):
        ihf[:, i] = ihf[:, i-1] + 0.5 * (heat_flux[:, i] + heat_flux[:, i+1]) * q['dz']
    return ihf

In [7]:
# Unit conversion factors
IHF_SCALE = (
    q['H']**2 * q['f0_d']
    / (2 * q['length'] * q['ideal_gas_constant'])
    * q['length']**4
    / (q['H'] * q['time']**2)
    / 25
)
U_SCALE = q['length'] / q['time']

# z-level selector on interior grid
zlevel = lambda z_km: np.argmin(np.abs(q['z_d'][1:-1] / 1000 - z_km))

## Load Data

In [8]:
print("Loading Holton-Mass data...")
real_data = np.load(HM_DATA_PATH)
real_data = real_data[:TIMESTEPS, 1, :]
zonal_wind = real_data[:, LEVEL]
real_ihf = ihf_function(real_data)

print("Loading Emulator data...")
save = np.load(EM_DATA_PATH)
save = save[:TIMESTEPS, :]
ihf = ihf_function(save)

print(f"Loaded {TIMESTEPS} timesteps.")

Loading Holton-Mass data...


FileNotFoundError: [Errno 2] No such file or directory: 'long_run_3mil.npy'

## Compute 2-D Histograms (log$_{10}$ density)

In [ ]:
density_u_ihf, u_edges, ihf_edges = np.histogram2d(
    zonal_wind, real_ihf[:TIMESTEPS, zlevel(30)], bins=100, density=True,
)
density_log = np.log10(density_u_ihf + 1e-5)

model_density_u_ihf, m_u_edges, m_ihf_edges = np.histogram2d(
    save[:TIMESTEPS, LEVEL], ihf[:TIMESTEPS, zlevel(30)], bins=100, density=True,
)
model_density_log = np.log10(model_density_u_ihf + 1e-5)

vmin_common = np.floor(min(np.min(density_log), np.min(model_density_log)))
vmax_common = np.ceil(max(np.max(density_log), np.max(model_density_log)))

# Mask negligible bins
threshold = -5
density_masked = np.where(density_log > threshold, density_log, np.nan)
model_density_masked = np.where(model_density_log > threshold, model_density_log, np.nan)

print(f"Log-density range: [{vmin_common}, {vmax_common}]")

## Plot: Steady-State Density

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
custom_levels = np.linspace(vmin_common, vmax_common, 11)

# --- Holton-Mass panel ---
rU30, rIHF30 = np.meshgrid(u_edges[:-1], ihf_edges[:-1], indexing='ij')

im1 = ax[0].contourf(
    rIHF30.T * IHF_SCALE,
    rU30.T * U_SCALE,
    density_masked.T,
    levels=custom_levels,
    vmin=vmin_common, vmax=vmax_common,
    cmap='RdBu_r', extend='both',
)
ax[0].set_xlabel(r"IHF(30 km) [K $\cdot$ m$^2$/s]", fontsize=20)
ax[0].set_ylabel(r'$U$(30 km) [m/s]', fontsize=20)
ax[0].set_title(r'Holton-Mass Steady State Density $\pi(x)$', fontweight='bold', fontsize=20)
ax[0].set_facecolor('white')
ax[0].set_ylim(-60, 90)
ax[0].set_xlim(-25000, 80000)
cbar1 = plt.colorbar(im1, ax=ax[0], shrink=1.2)
cbar1.set_label('log10(Count + 1e-10)', fontsize=14)

# --- Emulator panel ---
mU30, mIHF30 = np.meshgrid(m_u_edges[:-1], m_ihf_edges[:-1], indexing='ij')

im2 = ax[1].contourf(
    mIHF30.T * IHF_SCALE,
    mU30.T * U_SCALE,
    model_density_masked.T,
    levels=custom_levels,
    vmin=vmin_common, vmax=vmax_common,
    cmap='RdBu_r', extend='both',
)
ax[1].set_xlabel(r"IHF(30 km) [K $\cdot$ m$^2$/s]", fontsize=20)
ax[1].set_ylabel(r'$U$(30 km) [m/s]', fontsize=20)
ax[1].set_title(r'Emulator: Steady State Density $\pi(x)$', fontweight='bold', fontsize=20)
ax[1].set_facecolor('white')
ax[1].set_ylim(-60, 90)
ax[1].set_xlim(-25000, 80000)
cbar2 = plt.colorbar(im2, ax=ax[1], shrink=1.2)
cbar2.set_label('log10(Count + 1e-10)', fontsize=14)

plt.tight_layout(pad=1.0)
os.makedirs(os.path.dirname(os.path.abspath(SAVE_PATH)), exist_ok=True)
plt.savefig(SAVE_PATH)
print(f"Saved {SAVE_PATH}")
plt.show()